#### **IMPORT DE BIBLIOTECAS**

In [36]:
import pandas as pd 
import numpy as np

import seaborn as sns 
import matplotlib.pyplot as plt 

import warnings
warnings.filterwarnings('ignore')


import sweetviz as sv

from modelo_credito.config import PAY_AMT_COLUMNS, PAY_COLUMNS
from modelo_credito.data import (
    apply_treatment,
    compute_treatment_limits,
    save_limits,
    save_splits,
)



pd.set_option('display.max_columns', None)

sns.set()


#### **SEPARANDO DADOS EM TREINO, VALIDACAO E TESTE**

In [37]:
df = pd.read_csv('../data/00_bronze/dataset_trat.csv', sep=';')

In [38]:
# CRINADO DATASET DE TRINAMENTO
treino = (
    df 
        .query('SAFRA < 200604')
)

# CRIANDO DATASET DE VALIDACAO
validacao = (
    df 
        .query('SAFRA in [200604, 200605]')
)

# CRIANDO DATASET DE TESTE
teste = (
    df 
        .query('SAFRA in [200606, 200607]')
)


treino.to_parquet('../data/00_bronze/treino.parquet')
validacao.to_parquet('../data/00_bronze/validacao.parquet')
teste.to_parquet('../data/00_bronze/teste.parquet')


##### **RELATÓRIO COM VISÃO GERAL DOS DADOSS**

In [39]:
report = sv.analyze(treino, target_feat='default')
report.show_html('../reports/sweetviz_treino.html')

                                             |          | [  0%]   00:00 -> (? left)

findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfo

Report ../reports/sweetviz_treino.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


#### **TRATAMENTO INICIAL DAS VARIAVEIS**

BILL_AMT - Foram encontrados valores menores do que 0. A estratégia aqui é substituir estes valores por -1.

PAY: Foram encontrados valores muito discrepantes. A estratégia aqui é substituir estes valores pelo percentil 90.

PAY_AMT: Foram encontrados valores muito discrepantes. A estratégia aqui é substituir estes valores extremos pelo percentil 90.

Os limites são calculados **apenas no treino** e aplicados em treino, validação e teste.

In [42]:
limites = compute_treatment_limits(
    treino,
    PAY_COLUMNS + PAY_AMT_COLUMNS,
    quantile=0.90,
)

limites


In [43]:
treino_tratado = apply_treatment(treino, limites)
validacao_tratado = apply_treatment(validacao, limites)
teste_tratado = apply_treatment(teste, limites)

save_splits(treino_tratado, validacao_tratado, teste_tratado, "../data/01_prata")
save_limits(limites, "../data/01_prata/limites_tratamento.json")

print("treino", treino_tratado.shape, round(treino_tratado["default"].mean(), 4))
print("validacao", validacao_tratado.shape, round(validacao_tratado["default"].mean(), 4))
print("teste", teste_tratado.shape, round(teste_tratado["default"].mean(), 4))


treino (18000, 26) 0.2306
validacao (6000, 26) 0.2033
teste (6000, 26) 0.211
